# DA5401 A8 — Ensemble Learning for Bike Share Demand



In [20]:

import math
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime


from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.impute import SimpleImputer


import joblib
import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 42


## Part A — Data Loading and Preprocessing



### **Process**
1. **Data loading**: The dataset `hour.csv` was loaded from local storage or downloaded from UCI if missing.  
2. **Cleaning**: Dropped irrelevant columns (`instant`, `dteday`, `casual`, `registered`).  
3. **Datetime feature**: Constructed a proper timestamp combining date and hour for sorting.  
4. **Sorting & splitting**: Ensured time-aware ordering and used the final 20% of records as the test set (chronological split).  
5. **Feature engineering**:
   - **Numerical**: temperature, humidity, windspeed, etc.
   - **Categorical**: season, weather situation, month, hour, weekday, working day, year.
6. **Preprocessing pipeline**:
   - Numeric → median imputation + standard scaling  
   - Categorical → mode imputation + one-hot encoding  
   - Combined via `ColumnTransformer`.





In [21]:


csv_path = "hour.csv"


df = pd.read_csv(csv_path)
print('Loaded hour.csv — shape:', df.shape)
df.head()


Loaded hour.csv — shape: (17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [22]:

drop_cols = ['instant', 'dteday', 'casual', 'registered']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [23]:
# Create a proper datetime index using original dataset fields if available
orig_cols = pd.read_csv(csv_path, nrows=1).columns.tolist()

raw = pd.read_csv(csv_path, usecols=['dteday','hr'])
df['datetime'] = pd.to_datetime(raw['dteday'] + ' ' + raw['hr'].astype(str) + ':00:00')


# Sort by time 
df = df.sort_values('datetime').reset_index(drop=True)

# Target and features
y = df['cnt'].astype(float)
X = df.drop(columns=['cnt'])

print('Prepared X and y. X shape:', X.shape)

Prepared X and y. X shape: (17379, 13)


In [24]:


numeric_features = ['temp','atemp','hum','windspeed'] 
categorical_features = ['season','weathersit','mnth','hr','weekday','workingday','yr']

numeric_features, categorical_features


(['temp', 'atemp', 'hum', 'windspeed'],
 ['season', 'weathersit', 'mnth', 'hr', 'weekday', 'workingday', 'yr'])

In [25]:

# Preprocessor: impute numeric, scale; one-hot encode categoricals
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')


In [26]:

# Time-aware train/test split: last 20% as test
n = len(X)
test_size = int(0.2 * n)
train_idx = list(range(0, n - test_size))
test_idx = list(range(n - test_size, n))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]} (time-aware split)')


Train size: 13904, Test size: 3475 (time-aware split)



## Baseline Models

### **Models Used**
1. **Decision Tree Regressor** (`max_depth=6`)
2. **Linear Regression**

### **Purpose**
To establish benchmark RMSE values for later comparison with ensemble methods.

| Model | RMSE |
|--------|------|
| Decision Tree (depth=6) | **158.71** |
| Linear Regression | **133.85** |

### **Interpretation**
- The **Decision Tree** shows high variance (sensitive to training data) but low bias.
- The **Linear Regression** shows low variance but high bias (misses nonlinear patterns).  
These serve as baselines for variance-reduction (Bagging) and bias-reduction (Boosting) experiments.



In [12]:

# Baseline 1: Decision Tree Regressor (max_depth=6)
dt_pipeline = Pipeline(steps=[('pre', preprocessor), ('model', DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE))])
dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)
rmse_dt = math.sqrt(mean_squared_error(y_test, y_pred_dt))

# Baseline 2: Linear Regression
lr_pipeline = Pipeline(steps=[('pre', preprocessor), ('model', LinearRegression())])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
rmse_lr = math.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f'RMSE Decision Tree (depth=6): {rmse_dt:.4f}')
print(f'RMSE Linear Regression     : {rmse_lr:.4f}')

baseline_model = 'DecisionTree' if rmse_dt < rmse_lr else 'LinearRegression'
baseline_rmse = min(rmse_dt, rmse_lr)
print('Baseline chosen:', baseline_model, 'RMSE =', baseline_rmse)


RMSE Decision Tree (depth=6): 158.7148
RMSE Linear Regression     : 133.8546
Baseline chosen: LinearRegression RMSE = 133.85463492124302


## Part B.1 — Bagging (Variance Reduction)

### **Hypothesis**
> Bagging primarily reduces **variance** by aggregating predictions from multiple independent base learners.

### **Implementation**
- Base model: Decision Tree (depth=6)
- Ensemble: Bagging Regressor with 50 estimators (bootstrap sampling)

| Model | RMSE |
|-------|------|
| Decision Tree (depth=6) | 158.71 |
| Bagging (50 estimators) | **155.43** |

### **Observation**
- RMSE improved slightly, confirming **variance reduction**.
- The improvement is modest because the base trees were already shallow (low-variance).  
  Deeper trees would have shown stronger gains.

### **Interpretation**
Bagging reduced prediction instability by averaging multiple trees, smoothing over random fluctuations — a classic variance reduction effect.  
However, since the tree’s bias remained unchanged, the improvement was limited.



In [14]:

# Bagging with 50 base estimators (variance reduction test)
base_dt = DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE)
bagging = Pipeline(steps=[('pre', preprocessor), ('model', BaggingRegressor(estimator=base_dt, n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1))])
bagging.fit(X_train, y_train)
y_pred_bag = bagging.predict(X_test)
rmse_bag = math.sqrt(mean_squared_error(y_test, y_pred_bag))
print(f'RMSE Bagging (50 trees): {rmse_bag:.4f}')


RMSE Bagging (50 trees): 155.4306


## Part B.2 — Boosting (Bias Reduction)

### **Hypothesis**
> Boosting primarily reduces **bias** by sequentially training weak learners that correct each other’s residual errors.

### **Implementation**
- Model: `GradientBoostingRegressor`
- Tuned using grid search:
  - `n_estimators ∈ {100, 200}`
  - `learning_rate ∈ {0.05, 0.1}`
  - `max_depth ∈ {3, 4}`

| Model | RMSE |
|-------|------|
| Gradient Boosting (tuned) | **92.59** |
| Decision Tree (depth=6) | 158.71 |
| Bagging (50 estimators) | **155.43** |
| Linear Regression | **133.85** |


### **Observation**
- Large RMSE drop (≈40% improvement vs Bagging).  
- Boosting successively corrected residual errors, reducing bias dramatically.
- The model generalizes better while maintaining controlled variance.

### **Interpretation**
Boosting effectively balances bias and variance, making it the most powerful of the three ensemble methods for this task.


In [15]:

# Gradient Boosting with a small grid search using TimeSeriesSplit
gbr_pipeline = Pipeline(steps=[('pre', preprocessor), ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))])

param_grid = {'model__n_estimators': [100, 200], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 4]}
tscv = TimeSeriesSplit(n_splits=3)
gsearch = GridSearchCV(gbr_pipeline, param_grid, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
gsearch.fit(X_train, y_train)
best_gbr = gsearch.best_estimator_
y_pred_gbr = best_gbr.predict(X_test)
rmse_gbr = math.sqrt(mean_squared_error(y_test, y_pred_gbr))
print('Best GBR params:', gsearch.best_params_)
print(f'RMSE GradientBoosting: {rmse_gbr:.4f}')


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best GBR params: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 200}
RMSE GradientBoosting: 92.5926


## Part C — Stacking (Model Combination)

### **Hypothesis**
> Stacking combines **heterogeneous learners** to capture complementary decision boundaries.

### **Implementation**
- Base learners:
  - KNN Regressor  
  - Bagging Regressor  
  - Gradient Boosting Regressor  
- Meta-learner: Ridge Regression

| Model | RMSE |
|-------|------|
| Gradient Boosting (tuned) | **92.59** |
| Stacking (KNN + Bagging + GBR) | **116.79** |
| Bagging (50 estimators) | **155.43** |
| Linear Regression | **133.85** |

### **Observation**
- Stacking outperformed Bagging and Linear Regression but lagged behind Boosting.
- The improvement over Bagging demonstrates benefit from combining diverse learners.
- Ridge meta-learner likely underfitted slightly, limiting peak performance.

### **Interpretation**
Stacking offered moderate bias and variance control simultaneously, ranking second overall.


In [17]:


knn = KNeighborsRegressor(n_neighbors=8, n_jobs=-1)
bagging_for_stack = BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=6), n_estimators=50, random_state=RANDOM_STATE)
gbr_for_stack = GradientBoostingRegressor(random_state=RANDOM_STATE, n_estimators=200, learning_rate=0.05, max_depth=3)

estimators = [
    ('knn', knn),
    ('bagging', bagging_for_stack),
    ('gbr', gbr_for_stack)
]

stacking = Pipeline(steps=[('pre', preprocessor),
                           ('stack', StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1.0), n_jobs=-1, passthrough=False))])

stacking.fit(X_train, y_train)
y_pred_stack = stacking.predict(X_test)
rmse_stack = math.sqrt(mean_squared_error(y_test, y_pred_stack))
print(f'RMSE Stacking Regressor: {rmse_stack:.4f}')


RMSE Stacking Regressor: 116.7886


In [18]:

# Compile results into a neat table
results = pd.DataFrame([
    {'model': 'DecisionTree (depth=6)', 'rmse': rmse_dt},
    {'model': 'LinearRegression', 'rmse': rmse_lr},
    {'model': 'Bagging (50)', 'rmse': rmse_bag},
    {'model': 'GradientBoosting (tuned)', 'rmse': rmse_gbr},
    {'model': 'Stacking (KNN+Bagging+GBR)', 'rmse': rmse_stack}
]).sort_values('rmse').reset_index(drop=True)

results['rank'] = results['rmse'].rank(method='min')
results


,model,rmse,rank
0,GradientBoosting (tuned),92.592590,1.0
1,Stacking (KNN+Bagging+GBR),116.788580,2.0
2,LinearRegression,133.854635,3.0
3,Bagging (50),155.430553,4.0
4,DecisionTree (depth=6),158.714828,5.0


## Part D — Comparative Results

| Model | RMSE | Rank |
|--------|------|------|
| Gradient Boosting (tuned) | **92.59** | 1 |
| Stacking (KNN + Bagging + GBR) | **116.79** | 2 |
| Linear Regression | **133.85** | 3 |
| Bagging (50 estimators) | **155.43** | 4 |
| Decision Tree (depth=6) | **158.71** | 5 |

### **Observations**
- Bagging: Variance reduced, bias unchanged → small improvement.  
- Boosting: Bias reduced, variance controlled → large improvement.  
- Stacking: Combines model diversity → intermediate improvement.  
- Linear Regression: Low variance, high bias.  
- Decision Tree: High variance, low bias.  

These trends perfectly match ensemble learning theory.

---

##  Bias–Variance Trade-off Summary

| Technique | Bias | Variance | Typical Behavior | Observed Outcome |
|------------|------|-----------|------------------|------------------|
| Decision Tree | Low | High | Overfits | Worst RMSE |
| Bagging | Same Bias | Lower Variance | Smooths predictions | Minor improvement |
| Boosting | Lower Bias | Controlled Variance | Learns residuals | Best performance |
| Stacking | Medium Bias | Medium Variance | Combines models | Second best |
| Linear Regression | High Bias | Low Variance | Underfits | Moderate RMSE |

**Summary:**  
- Bagging confirms *variance reduction*.  
- Boosting confirms *bias reduction*.  
- Stacking validates *model diversity advantage*.

---

### Why the Stacking Regressor (Best Ensemble) Outperformed the Single Model Baseline

The **Stacking Regressor** outperformed the single-model baseline because it leverages **model diversity** and optimally balances the **bias–variance trade-off** through hierarchical learning.

#### 1️⃣ Bias–Variance Trade-off Perspective
A single model (like a Decision Tree or Linear Regression) is limited by its inherent bias–variance characteristics:
- **Decision Tree** → low bias, high variance (unstable).
- **Linear Regression** → high bias, low variance (over-simplified).

Stacking reduces both problems by **combining models that occupy different regions of the bias–variance spectrum**.  
- The **high-variance** learners (trees, KNN) bring flexibility.  
- The **low-variance** learner (Ridge meta-model) smooths and stabilizes their predictions.  
- The final ensemble averages and re-weights base predictions to minimize total generalization error.

The ensemble prediction can be written as the meta-function of base predictions:

$$
\hat{y}_{\text{stack}} = f\!\big(w_1 \hat{y}_1,\; w_2 \hat{y}_2,\; \dots,\; w_k \hat{y}_k\big)
$$

This typically reduces expected error because correlated errors among base models are averaged out. In compact form:

$$
\operatorname{Var}(\hat{y}_{\text{stack}}) < \operatorname{Var}(\hat{y}_i)
\qquad\text{and}\qquad
\operatorname{Bias}(\hat{y}_{\text{stack}}) < \operatorname{Bias}(\text{high-bias model})
$$


#### 2️⃣ Model Diversity Perspective
Stacking intentionally combines **heterogeneous models** — KNN, Bagging, and Gradient Boosting — each capturing different structural patterns in the data:
- **KNN** captures local neighborhood effects.
- **Bagging** stabilizes high-variance tree predictions.
- **Gradient Boosting** captures residual, global patterns.

The **meta-learner (Ridge Regression)** learns optimal weights for these diverse base learners, emphasizing models that perform well in certain data regions and de-emphasizing others.  
This **diversity in hypothesis space** reduces overfitting risk and enhances generalization.

#### 3️⃣ Empirical Evidence
| Model | RMSE |
|--------|------|
| Linear Regression (baseline) | 133.85 |
| Stacking (KNN + Bagging + GBR) | **116.79** |

The Stacking Regressor achieved ~13% lower RMSE than the single-model baseline, confirming:
- Variance was reduced via ensemble averaging.
- Bias was reduced via complementary model learning.
- Model diversity improved robustness against dataset noise and feature interactions.

#### ✅ Conclusion
The **Stacking Regressor** (and ensembles in general) outperform single models because they **combine multiple perspectives on the data**, achieving a **better bias–variance balance** and **leveraging model diversity** to generalize more effectively to unseen examples.

